In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con).to_string(index=False))

In [ ]:
q1 = "SELECT COUNT(*) AS rows_total, SUM(analysed) AS analysed FROM matches"
print(pd.read_sql(q1, con).to_string(index=False))

q2 = """
SELECT t.name, COUNT(*) AS matches
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1 AND m.file_year >= 2015
GROUP BY t.name
ORDER BY matches DESC
"""
print(pd.read_sql(q2, con).to_string(index=False))

In [ ]:
#matches per year and per surface
q3 = """
SELECT file_year, surface, COUNT(*) AS matches
FROM matches
WHERE file_year >= 2015
GROUP BY file_year, surface
ORDER BY file_year, surface
"""
print(pd.read_sql(q3, con).to_string(index=False))

In [ ]:
# same as above but in a table 
df3 = pd.read_sql(q3, con)
print(df3.pivot(index="file_year", columns="surface", values="matches").to_string())

In [ ]:
#percentage of service points that ended in ace
q4 = """
SELECT t.name,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name
ORDER BY aces_pct DESC
"""
print(pd.read_sql(q4, con).to_string(index=False))

In [ ]:
#same above but evolution by years
q5 = """
SELECT t.name,
       m.file_year,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name, m.file_year
HAVING COUNT(*) >= 20
ORDER BY t.name, m.file_year
"""
df5 = pd.read_sql(q5, con)

print(df5.pivot(index="name", columns="file_year", values="aces_pct").to_string())
print()
print(df5.pivot(index="name", columns="file_year", values="matches_used").to_string())

In [ ]:
con.execute("DROP VIEW IF EXISTS serve_obs")
con.execute("""
CREATE VIEW serve_obs AS
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       winner_id AS server_id, loser_id AS returner_id,
       w_svpt AS svpt, w_1stWon + w_2ndWon AS sv_won, w_ace AS ace
FROM matches
WHERE retired = 0
  AND w_svpt IS NOT NULL AND w_1stWon IS NOT NULL AND w_2ndWon IS NOT NULL
  AND w_ace IS NOT NULL AND w_svpt > 0
UNION ALL
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       loser_id, winner_id,
       l_svpt, l_1stWon + l_2ndWon, l_ace
FROM matches
WHERE retired = 0
  AND l_svpt IS NOT NULL AND l_1stWon IS NOT NULL AND l_2ndWon IS NOT NULL
  AND l_ace IS NOT NULL AND l_svpt > 0
""")
con.commit()

In [ ]:
print(pd.read_sql("""
SELECT COUNT(*) AS rows_total,
       SUM(analysed) AS analysed_rows,
       SUM(CASE WHEN sv_won > svpt OR ace > svpt OR ace > sv_won THEN 1 ELSE 0 END) AS impossible_rows
FROM serve_obs
""", con).to_string(index=False))

print(pd.read_sql("""
SELECT surface,
       COUNT(*) AS rows_n,
       ROUND(100.0 * SUM(sv_won) / SUM(svpt), 2) AS service_pts_won_pct,
       ROUND(100.0 * SUM(ace) / SUM(svpt), 2) AS aces_pct
FROM serve_obs
WHERE file_year >= 2015
GROUP BY surface
ORDER BY service_pts_won_pct DESC
""", con).to_string(index=False))

In [ ]:
import numpy as np

obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
tournament_names = pd.read_sql("SELECT event_no, name FROM tournaments", con)
print(obs.shape)


def window_sums(df, keys, prefix):
    base = df.groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    parts = [base.assign(file_year=base["file_year"] + d) for d in (-1, 0, 1)]
    out = pd.concat(parts).groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    return out.rename(columns={"svpt": prefix + "_svpt", "val": prefix + "_val"})


def role_excess(t, d, pid_col, sf, prefix, mu, k):
    dd = d.assign(pid=d[pid_col])
    tt = t.assign(pid=t[pid_col])
    w = window_sums(dd, sf + ["pid"], prefix + "w")
    e = window_sums(dd, sf + ["pid", "event_no"], prefix + "e")
    tt = tt.merge(w, on=sf + ["pid", "file_year"], how="left")
    tt = tt.merge(e, on=sf + ["pid", "event_no", "file_year"], how="left")
    n = (tt[prefix + "w_svpt"].fillna(0) - tt[prefix + "e_svpt"].fillna(0)).to_numpy()
    v = (tt[prefix + "w_val"].fillna(0) - tt[prefix + "e_val"].fillna(0)).to_numpy()
    return (v - mu * n) / (n + k), n


def expected_values(obs, value, by_surface=True, k=500, analysed_only=True):
    sf = ["surface"] if by_surface else []
    d = obs.rename(columns={value: "val"})[["match_id", "file_year", "event_no", "analysed", "surface",
                                              "server_id", "returner_id", "svpt", "val"]]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, sf, "wt"), on=sf + ["file_year"], how="left")
    t = t.merge(window_sums(d, sf + ["event_no"], "et"), on=sf + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", sf, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", sf, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_spw = expected_values(obs, "sv_won")
obs_pct = 100 * res_spw["val"].sum() / res_spw["svpt"].sum()
exp_pct = 100 * (res_spw["exp"] * res_spw["svpt"]).sum() / res_spw["svpt"].sum()
print(len(res_spw), "rows | observed", round(float(obs_pct), 2), "% | expected", round(float(exp_pct), 2), "%")
print("rows with no server baseline:", int((res_spw["n_srv"] == 0).sum()),
      "| no returner baseline:", int((res_spw["n_ret"] == 0).sum()))

In [ ]:
def speed_table(res, B=1000, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for ev, g in m.groupby("event_no"):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "matches": len(g),
                    "speed_pts": round(100 * r.sum() / s.sum(), 2),
                    "ci_low": round(float(lo), 2), "ci_high": round(float(hi), 2)})
    return (pd.DataFrame(out).merge(tournament_names, on="event_no")
            .sort_values("speed_pts", ascending=False)[["name", "matches", "speed_pts", "ci_low", "ci_high"]])

print("Service points won, relative to other events on the same surface")
print(speed_table(res_spw).to_string(index=False))

In [ ]:
res_ace = expected_values(obs, "ace")
print("Aces, relative to other events on the same surface")
print(speed_table(res_ace).to_string(index=False))
print()
res_spw_all = expected_values(obs, "sv_won", by_surface=False)
print("Service points won, relative to ALL other events (surfaces mixed)")
print(speed_table(res_spw_all).to_string(index=False))

In [ ]:
res_all = expected_values(obs, "sv_won", analysed_only=False)
o = 100 * res_all["val"].sum() / res_all["svpt"].sum()
e = 100 * (res_all["exp"] * res_all["svpt"]).sum() / res_all["svpt"].sum()
print(len(res_all), "rows | observed", round(float(o), 2), "% | expected", round(float(e), 2), "%")

In [ ]:
t3 = speed_table(res_spw_all)
print(t3.iloc[:7].to_string(index=False))
print(t3.iloc[7:].to_string(index=False))

In [ ]:
obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface, indoor,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
print(obs.shape, obs["indoor"].value_counts(dropna=False).to_dict())

In [ ]:
def expected_values_k(obs, value, keys, k=500, analysed_only=True):
    base_cols = ["match_id", "file_year", "event_no", "analysed", "server_id", "returner_id", "svpt"]
    d = obs.rename(columns={value: "val"})[list(dict.fromkeys(base_cols + keys + ["val"]))]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, keys, "wt"), on=keys + ["file_year"], how="left")
    t = t.merge(window_sums(d, keys + ["event_no"], "et"), on=keys + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", keys, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", keys, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_roof = expected_values_k(obs, "sv_won", ["surface", "indoor"])
print(len(res_roof), "rows | observed",
      round(float(100 * res_roof["val"].sum() / res_roof["svpt"].sum()), 2), "% | expected",
      round(float((100 * res_roof["exp"] * res_roof["svpt"]).sum() / res_roof["svpt"].sum()), 2), "%")
print("Service points won, relative to events with the same surface and roof")
print(speed_table(res_roof).to_string(index=False))

In [ ]:
cols = []
for k in [100, 250, 500, 1000, 2000]:
    r_ = expected_values_k(obs, "sv_won", ["surface", "indoor"], k=k)
    cols.append(speed_table(r_, B=100).set_index("name")["speed_pts"].rename(f"k={k}"))
print(pd.concat(cols, axis=1).round(2).to_string())

In [ ]:
def speed_by_year(res, B=500, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no", "file_year"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for (ev, yr), g in m.groupby(["event_no", "file_year"]):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "year": yr, "matches": len(g),
                    "speed_pts": 100 * r.sum() / s.sum(), "half_width": (hi - lo) / 2})
    return pd.DataFrame(out).merge(tournament_names, on="event_no")

by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())

In [ ]:
by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())